# Mutual Fund Exploratory Analysis

Interactive exploration on top of the same engine the pipeline and dashboard use
(`src.analyzer` / `src.metrics`), so anything you find here matches the published report.

Run `python main_pipeline.py` first to populate `data/mf_database.db`.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable whether the notebook runs from / or /notebooks.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src import analyzer, config, db_manager, metrics

DB = ROOT / "data" / "mf_database.db"
catalogue = db_manager.scheme_catalogue(DB)
catalogue

## Run the analysis

`analyse()` validates the data first and excludes any scheme with a critical finding, so a
missing scheme is a data problem, not a bug — check `result.quality` when one disappears.

In [ ]:
codes = catalogue["scheme_code"].astype(str).tolist()

result = analyzer.analyse(DB, scheme_codes=codes, benchmark_code=config.BENCHMARK_SCHEME)

for insight in result.insights:
    print(f"{insight.rank}. [{insight.category}] {insight.headline}\n   {insight.detail}\n")

result.to_frame()

## Data quality first

Never interpret a return before checking what produced it.

In [ ]:
print(result.quality.summary())
pd.DataFrame(
    [
        {"severity": f.severity, "scheme": f.scheme_code, "check": f.check, "detail": f.message}
        for f in result.quality.sorted_findings()
    ]
)

## Growth of a common investment

Raw NAVs are not comparable across schemes (a ₹450 NAV is not "expensive"). Rebasing every
series to the same amount from a shared start date is the only fair visual comparison.

In [ ]:
series = {s.scheme_name: result.nav_series[s.scheme_code] for s in result.schemes}
start = max(v.index[0] for v in series.values())

fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
for name, nav in series.items():
    window = nav.loc[start:]
    axes[0].plot(window.index, metrics.growth_of(window, 10_000), label=name, linewidth=1.6)
    axes[1].plot(window.index, metrics.drawdown_series(window), label=name, linewidth=1.2)

axes[0].set_title(f"Growth of Rs 10,000 invested on {start:%d %b %Y}")
axes[0].set_ylabel("Value (Rs)")
axes[0].legend(fontsize=8)
axes[0].grid(True, linestyle="--", alpha=0.4)

axes[1].set_title("Drawdown from running peak")
axes[1].set_ylabel("Drawdown (%)")
axes[1].axhline(0, color="grey", linewidth=0.8)
axes[1].grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

## Risk vs return

The scatter that matters: anything up and to the left earned more per unit of volatility.

In [ ]:
frame = result.to_frame().dropna(subset=["volatility_pct", "cagr_3y_pct"])

plt.figure(figsize=(9, 6))
plt.scatter(frame["volatility_pct"], frame["cagr_3y_pct"], s=90)
for row in frame.itertuples():
    plt.annotate(
        row.scheme_name[:34],
        (row.volatility_pct, row.cagr_3y_pct),
        textcoords="offset points",
        xytext=(6, 5),
        fontsize=8,
    )
plt.axhline(
    result.assumptions["risk_free_rate_annual_pct"],
    color="crimson",
    linestyle="--",
    label="Risk-free rate",
)
plt.xlabel("Annualised volatility (%)")
plt.ylabel("3Y CAGR (%)")
plt.title("Risk vs return")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

## Rolling 3-year returns

Point-to-point returns are hostage to their start date. The rolling distribution shows whether
a fund was consistently good or merely lucky in one window.

In [ ]:
pd.DataFrame(
    [
        {
            "scheme": s.scheme_name,
            "windows": s.rolling_3y.observations,
            "worst %": round(s.rolling_3y.min_pct, 2),
            "median %": round(s.rolling_3y.median_pct, 2),
            "best %": round(s.rolling_3y.max_pct, 2),
            "% positive": round(s.rolling_3y.positive_share_pct, 1),
        }
        for s in result.schemes
        if s.rolling_3y
    ]
)

## Assumptions

Every number above depends on these. Change the risk-free rate and the Sharpe ranking can change.

In [ ]:
pd.DataFrame(result.assumptions.items(), columns=["assumption", "value"])